<style>
/* Persian text styling */
.jp-RenderedHTMLCommon, .rendered_html, .markdown-body, .jp-MarkdownOutput {
    font-family: Vazir, Tahoma, sans-serif !important;
}
</style>

<div dir="rtl" style="text-align: center; padding: 20px; font-family: Vazir, Tahoma, sans-serif;">
  <h1 style="margin: 0; font-size: 30px;">
    تکلیف ۱: مدل‌سازی زبانی برای «رمزگشایی سیگنال»
  </h1>
  <p style="margin-top: 12px; font-size: 16px; color: #555;">
    n-gram Language Models • Character-level Prediction • NLP
  </p>
</div>

## <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">اطلاعات کلی</div>

<div dir="rtl" style="text-align: right; line-height: 2; padding: 18px; background-color: #0f172a; color: #f8fafc; border-radius: 14px; font-family: Vazir, Tahoma, sans-serif;">

<b>مهلت تحویل:</b> توسط مدرس اعلام می‌شود.<br>
<b>شیوه‌ی تحویل:</b> ارسال فایل کامل همین نوت‌بوک.<br>
<b>نحوه‌ی نمره‌دهی:</b> بخش اصلی نمره بر اساس درستی پیاده‌سازی‌ها است و بخشی هم به خوانایی و کارایی کد اختصاص دارد.<br>
<b>نسخه‌ی پیشنهادی پایتون:</b> Python 3.11

<hr style="border-color: #334155;">

<b>اجرای محلی (Local):</b><br>
برای اجرای این تکلیف روی سیستم شخصی، فقط این موارد را لازم دارید:
<ul style="margin-top: 8px;">
  <li>Python 3.11</li>
  <li>Jupyter Notebook یا JupyterLab</li>
  <li>فایل نوت‌بوک</li>
  <li>فایل داده‌ی همراه با نام <code>hw1_word_bank.txt</code> در همان پوشه‌ی نوت‌بوک</li>
</ul>

سپس نوت‌بوک را باز کنید و مطمئن شوید فایل <code>hw1_word_bank.txt</code> کنار آن قرار دارد. در این نسخه هیچ دانلود اضافه‌ای در زمان اجرا لازم نیست و همه‌چیز با کتابخانه‌های استاندارد پایتون انجام می‌شود.
</div>

# <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">مرور کلی</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
در این تکلیف قرار است یک عامل خودکار برای یک سناریوی «رمزگشایی سیگنال» بسازید. در این سناریو یک واژه‌ی پنهان داریم که فقط طول آن و بخشی از حروف حدس‌زده‌شده‌اش دیده می‌شود. عامل شما باید در هر مرحله یک حرف پیشنهاد دهد تا واژه با کمترین خطا بازیابی شود.

برای این کار چند راهبرد مختلف بر پایه‌ی مدل‌های زبانی حرف‌محور از نوع <b>n-gram</b> پیاده‌سازی می‌کنید. هدف نهایی این است که مدلی بسازید که با کمترین تعداد حدس اشتباه، واژه‌های مخفی را بازسازی کند.

برای ساده‌تر شدن اجرا، فایل واژه‌های موردنیاز همراه تکلیف قرار داده شده است و در این نسخه به هیچ دانلود بیرونی یا کتابخانه‌ی اضافی برای داده نیاز ندارید.
</div>

# <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سامانه‌ی رمزگشایی سیگنال (۷ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
<b>توضیح سناریو:</b> فرض کنید یک کانال ارتباطی قدیمی فقط یک واژه‌ی کلیدی را مخابره می‌کند، اما بخشی از نویز کانال باعث شده فقط جایگاه حروف مشخص باشد و خود حروف پنهان شوند. شما در هر مرحله تنها مجاز هستید <b>یک حرف</b> پیشنهاد دهید. اگر حرف درست باشد، تمام رخدادهای آن در واژه آشکار می‌شود. اگر غلط باشد، یکی از فرصت‌های شما کم می‌شود. بازی وقتی تمام می‌شود که یا کل واژه بازیابی شود، یا تعداد خطاها از حد مجاز عبور کند.

در ادامه، یک پیاده‌سازی ساده از این فرایند آمده است. <b>در این بخش نیازی به پیاده‌سازی ندارید.</b>
</div>

In [1]:
def decode_signal(secret_word, decoder, max_errors=8, verbose=True, **decoder_args):
    """
    Run the masked-word decoding game and return the number of wrong guesses.

    secret_word: target word in lowercase letters
    decoder: function that proposes the next character
        inputs:
            mask: current known pattern, with "_" for unknown positions
            guessed: set of characters already proposed
            decoder_args: optional extra keyword arguments
    max_errors: maximum number of wrong guesses allowed
    verbose: whether to print step-by-step details
    decoder_args: extra keyword arguments passed to the decoder
    """
    secret_word = secret_word.lower()
    mask = ["_"] * len(secret_word)
    guessed = set()

    if verbose:
        print("Signal decoding started")
        print("Target pattern:", " ".join(mask), "| Word length:", len(secret_word))

    errors = 0

    while errors < max_errors:
        if verbose:
            print("Remaining allowed errors:", max_errors - errors)

        guess = decoder(mask, guessed, **decoder_args)

        if verbose:
            print("Proposed character:", guess)

        if guess in guessed:
            if verbose:
                print("This character was already tried.")
            errors += 1
        else:
            guessed.add(guess)

            if guess in secret_word and len(guess) == 1:
                for index, char in enumerate(secret_word):
                    if char == guess:
                        mask[index] = char
                if verbose:
                    print("Correct guess:", " ".join(mask))
            else:
                if len(guess) != 1:
                    print("Please enter exactly one character.")
                if verbose:
                    print("Wrong guess.")
                errors += 1

        if "_" not in mask:
            if verbose:
                print("Decoding completed successfully.")
            return errors

    if verbose:
        print("No attempts left. The original word was:", secret_word)
    return errors


<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
در اینجا یک حدس‌زن دستی برای بازی تعاملی داریم.
</div>

In [2]:
def manual_decoder(mask, guessed, **kwargs):
    """
    A simple function for manual play.
    """
    print("\nEnter the next character:")
    return input().lower().strip()


<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
اگر می‌خواهید بازی را به‌صورت تعاملی اجرا کنید، مقدار <code>interactive</code> را <code>True</code> قرار دهید. برای تحویل نهایی، آن را روی <code>False</code> بگذارید تا اجرای کامل نوت‌بوک خودکار بماند.
</div>

In [3]:
interactive = False

<div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;"><b>برای آزمون دستی:</b></div>

In [4]:
if interactive:
    decode_signal("whatever", manual_decoder, 8, True)

### <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سؤال ۱ (۱ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
در این نسخه، یک فایل داده‌ی متنی با نام <code>hw1_word_bank.txt</code> همراه تکلیف قرار داده شده است. در هر خط از این فایل یک واژه‌ی انگلیسی آمده است. در این بخش باید واژه‌ها را از فایل بخوانید و <b>نوع‌های واژگانی یکتا</b> را به‌دست آورید، با این شرط که فقط واژه‌هایی را نگه دارید که کاملاً از حروف الفبا تشکیل شده باشند. همه‌ی واژه‌ها باید به حروف کوچک تبدیل شوند.

سپس این مجموعه‌ی واژه‌ها را به‌صورت تصادفی با <code>random.shuffle</code> به‌هم بزنید و آن را به دو بخش جدا تقسیم کنید:
<ul>
  <li><b>مجموعه‌ی ارزیابی</b>: دقیقاً 1000 واژه</li>
  <li><b>مجموعه‌ی آموزش</b>: بقیه‌ی واژه‌ها</li>
</ul>

دقت کنید که واژه‌های مجموعه‌ی ارزیابی نباید در مجموعه‌ی آموزش باشند. هدف این است که عامل شما روی واژه‌هایی سنجیده شود که مستقیماً آن‌ها را در آموزش ندیده است.

<b>وظیفه:</b> دو لیست <code>train_words</code> و <code>eval_words</code> را بسازید.<br>
<b>بررسی:</b> از assertionهای پایین استفاده کنید.
</div>

In [5]:
from pathlib import Path
import random
import re

random.seed(1)

WORD_BANK_PATH = Path("hw1_word_bank.txt")
if not WORD_BANK_PATH.exists():
    raise FileNotFoundError(
        "Could not find 'hw1_word_bank.txt'. Put the data file next to this notebook."
    )

# train_words stores word types used for training
train_words = []

# eval_words stores 1000 word types used for evaluation
eval_words = []


###
# Your answer BEGINS HERE
###

# read the words from given file
with open(WORD_BANK_PATH, "r", encoding="utf-8") as f:
    lines = f.read().splitlines()

# finding unique words by lowercasing all lines (words) and checking the words consist of only alphabets
unique_words = set()
for line in lines:
    word = line.strip().lower()
    if word.isalpha():
        unique_words.add(word)

# turn into a list and shuffle
word_list = list(unique_words)
random.shuffle(word_list)

# train and eval split
eval_words = word_list[:1000]
train_words = word_list[1000:]

###
# Your answer ENDS HERE
###

print("Number of word types in eval =", len(eval_words))
print("Number of word types in train =", len(train_words))


Number of word types in eval = 1000
Number of word types in train = 41000


<div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;"><b>برای آزمون شما:</b></div>

In [6]:
assert(len(train_words) > 35000 and len(train_words) < 45000)
assert(len(eval_words) == 1000)

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
<b>آزمایش تعاملی:</b><br>
می‌توانید یک واژه‌ی تصادفی از مجموعه‌ی ارزیابی را خودتان حدس بزنید. این بخش فقط برای تجربه‌ی بازی است؛ قبل از تحویل، <code>interactive</code> را روی <code>False</code> بگذارید.
</div>

In [7]:
# play the decoding game with a random evaluation word
if interactive:
    decode_signal(random.choice(eval_words), manual_decoder, 8, True)


### <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سؤال ۲ (۰.۵ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
برای داشتن یک خط مبنا، ابتدا یک روش بسیار ساده و کاملاً تصادفی پیاده‌سازی می‌کنید. این روش باید از میان حروف <code>a</code> تا <code>z</code>، یکی را به‌صورت تصادفی انتخاب کند؛ البته حروفی که قبلاً حدس زده شده‌اند نباید دوباره انتخاب شوند.

برای ارزیابی کیفیت این روش و روش‌های بعدی، تابع <code>evaluate_decoder</code> در اختیار شما قرار داده شده است. این تابع میانگین تعداد حدس‌های اشتباه را روی تمام واژه‌های مجموعه‌ی ارزیابی محاسبه می‌کند.

<b>وظیفه:</b> تابع <code>random_decoder</code> را کامل کنید.<br>
<b>بررسی:</b> از assertionهای پایین استفاده کنید.
</div>

In [8]:
def evaluate_decoder(decoder, evaluation_words):
    """
    Measure the average number of wrong guesses made by a decoder
    over all words in the evaluation set.
    """
    total_errors = 0
    for word in evaluation_words:
        total_errors += decode_signal(word, decoder, 26, False)
    return total_errors / float(len(evaluation_words))

In [9]:
import string
import random

def random_decoder(mask, guessed, **kwargs):

    ###
    # Your answer BEGINS HERE
    ###

    # create the set of english characters
    alphabet = set(string.ascii_lowercase)
    
    # characters not chosen
    available_letters = list(alphabet - guessed)
    
    # choosing a random character
    return random.choice(available_letters)

    ###
    # Your answer ENDS HERE
    ###

sample_word = random.choice(eval_words)
print("Selected word =", sample_word)
print("Mistakes made by random decoder =", decode_signal(sample_word, random_decoder, 26, False))

result = evaluate_decoder(random_decoder, eval_words)
print("\nEvaluating the random decoder on the full evaluation set")
print("Average number of incorrect guesses:", result)


Selected word = discloses
Mistakes made by random decoder = 13

Evaluating the random decoder on the full evaluation set
Average number of incorrect guesses: 16.7


<div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;"><b>برای آزمون شما:</b></div>

In [10]:
assert(result > 10 and result < 20)

### <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سؤال ۳ (۱ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
اکنون اولین مدل واقعی خود را می‌سازید: یک <b>مدل زبانی یونی‌گرام</b> در سطح حرف. در این مدل، باید فراوانی هر حرف را در کل واژه‌های آموزشی محاسبه کنید. سپس حدس‌زن شما باید حرفی را برگرداند که بر اساس این توزیع، بیشترین احتمال را دارد؛ البته حروفی که قبلاً حدس زده شده‌اند باید کنار گذاشته شوند.

<b>وظیفه:</b> فراوانی حروف را در متغیر <code>char_unigram_counts</code> ذخیره کنید و سپس تابع <code>unigram_decoder</code> را کامل کنید.<br>
<b>بررسی:</b> از assertionهای پایین استفاده کنید.
</div>

In [11]:
from collections import Counter
import string

char_unigram_counts = Counter()

###
# Your answer BEGINS HERE
###

# count the occurance of each character
for word in train_words:
    char_unigram_counts.update(word)
    
# make sure all characters have at least 0 occurance
for char in string.ascii_lowercase:
    if char not in char_unigram_counts:
        char_unigram_counts[char] = 0

###
# Your answer ENDS HERE
###

def unigram_decoder(mask, guessed, char_unigram_counts=char_unigram_counts):

    ###
    # Your answer BEGINS HERE
    ###

    best_char = None
    max_count = -1
    
    # search in all loswercases
    for char in string.ascii_lowercase:
        # if not guessed
        if char not in guessed:
            # check if character has highest number of occurance
            if char_unigram_counts[char] > max_count:
                max_count = char_unigram_counts[char]
                best_char = char
                
    return best_char

###
# Your answer ENDS HERE
###

result = evaluate_decoder(unigram_decoder, eval_words)
print("Evaluating the unigram decoder on the full evaluation set")
print("Average number of incorrect guesses:", result)

Evaluating the unigram decoder on the full evaluation set
Average number of incorrect guesses: 10.358


<div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;"><b>برای آزمون شما:</b></div>

In [12]:
assert(result > 5 and result < 15)

### <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سؤال ۴ (۱ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
طول واژه می‌تواند یک سرنخ مفید باشد. برای مثال، توزیع حروف در واژه‌های کوتاه و بلند دقیقاً یکسان نیست. در این بخش باید یک مدل یونی‌گرام بسازید که به <b>طول واژه</b> شرطی شده باشد؛ یعنی برای هر طول، یک توزیع جداگانه از حروف داشته باشید.

در زمان آزمون ممکن است با طولی روبه‌رو شوید که در آموزش ندیده‌اید. در این حالت، مدل شما باید به مدل یونی‌گرام ساده‌ی سؤال قبل برگردد.

<b>وظیفه:</b> فراوانی حروف شرطی‌شده بر طول را در <code>char_unigram_by_length</code> ذخیره کنید و سپس تابع <code>length_aware_unigram_decoder</code> را کامل کنید.<br>
<b>بررسی:</b> از assertionهای پایین استفاده کنید.
</div>

In [13]:
char_unigram_by_length = {}

###
# Your answer BEGINS HERE
###

# classify and count based on word length
for word in train_words:
    length = len(word)
    if length not in char_unigram_by_length:
        char_unigram_by_length[length] = Counter()
    char_unigram_by_length[length].update(word)

# make sure all chars have at least 0 appearance
for length in char_unigram_by_length:
    for char in string.ascii_lowercase:
        if char not in char_unigram_by_length[length]:
            char_unigram_by_length[length][char] = 0

###
# Your answer ENDS HERE
###

def length_aware_unigram_decoder(
    mask,
    guessed,
    char_unigram_by_length=char_unigram_by_length,
    char_unigram_counts=char_unigram_counts,
):
    ###
    # Your answer BEGINS HERE
    ###

    word_length = len(mask)
    
    # check length to use correct way
    if word_length in char_unigram_by_length:
        current_counts = char_unigram_by_length[word_length]
    else:
        current_counts = char_unigram_counts
        
    best_char = None
    max_count = -1
    
    # search for best character
    for char in string.ascii_lowercase:
        if char not in guessed:
            # used get to prevent key error
            count = current_counts.get(char, 0)
            if count > max_count:
                max_count = count
                best_char = char
                
    return best_char

    ###
    # Your answer ENDS HERE
    ###

result = evaluate_decoder(length_aware_unigram_decoder, eval_words)
print("Evaluating the length-aware unigram decoder on the full evaluation set")
print("Average number of incorrect guesses:", result)

Evaluating the length-aware unigram decoder on the full evaluation set
Average number of incorrect guesses: 10.343


<div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;"><b>برای آزمون شما:</b></div>

In [14]:
assert(result > 5 and result < 15)

### <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سؤال ۵ (۱ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
در این بخش باید یک <b>مدل بی‌گرام</b> روی حروف بسازید. ترتیب حروف در واژه بسیار مهم است و مدل‌های قبلی این موضوع را در نظر نمی‌گرفتند. برای نمونه، اگر بخشی از الگو به صورت <code>n _ s s</code> باشد، احتمال اینکه جای خالی با حرف <code>e</code> پر شود زیاد است.

مدل بی‌گرام شما باید برای هر جای خالی از <b>حرف سمت چپ</b> به عنوان متن زمینه استفاده کند. برای مثال، در الگوی <code>e _ c _ b _ _</code> برای سه جای خالی اول، متن زمینه‌ی چپ مشخص است؛ اما برای آخرین جای خالی چنین چیزی نداریم. در این حالت می‌توانید به مدل یونی‌گرام برگردید.

هنگام آموزش، ابتدای واژه را هم درست مدیریت کنید؛ مثلاً با یک نماد ویژه مانند <code>$</code>. اینکه آیا به نماد پایان واژه هم نیاز دارید یا نه، بخشی از طراحی شماست.

<b>وظیفه:</b> شمارش‌های بی‌گرام را در <code>char_bigram_counts</code> ذخیره کنید و سپس تابع <code>bigram_decoder</code> را کامل کنید.
</div>

In [15]:
from collections import defaultdict

char_bigram_counts = defaultdict(Counter)

###
# Your answer BEGINS HERE
###

# bi-gram train
for word in train_words:
    # start of word
    padded_word = "$" + word
    for i in range(len(padded_word) - 1):
        left_char = padded_word[i]
        curr_char = padded_word[i+1]
        char_bigram_counts[left_char][curr_char] += 1
        
###
# Your answer ENDS HERE
###

def bigram_decoder(mask, guessed, char_bigram_counts=char_bigram_counts, char_unigram_counts=char_unigram_counts):
    ###
    # Your answer BEGINS HERE
    ###
    best_char = None
    max_score = -1
    
    # calcualte score based on left char
    for char in string.ascii_lowercase:
        if char in guessed:
            continue
            
        score = 0
        has_context = False
        
        for i in range(len(mask)):
            if mask[i] == '_':
                if i == 0:
                    left_char = '$'
                elif mask[i-1] != '_':
                    left_char = mask[i-1]
                else:
                    left_char = None
                    
                if left_char is not None:
                    has_context = True
                    score += char_bigram_counts[left_char].get(char, 0)

        if has_context and score > max_score:
            max_score = score
            best_char = char

    # back to unigram if no context available
    if max_score <= 0:
        max_unigram_count = -1
        for char in string.ascii_lowercase:
            if char not in guessed:
                count = char_unigram_counts.get(char, 0)
                if count > max_unigram_count:
                    max_unigram_count = count
                    best_char = char
                    
    return best_char
    ###
    # Your answer ENDS HERE
    ###



result = evaluate_decoder(bigram_decoder, eval_words)
print("Evaluating the bigram decoder on the full evaluation set")
print("Average number of incorrect guesses:", result)

Evaluating the bigram decoder on the full evaluation set
Average number of incorrect guesses: 8.949


### <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سؤال ۶ (۲ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
اکنون نوبت طراحی یک عامل بهتر است. در این بخش می‌توانید خلاقانه‌تر عمل کنید. برای مثال:
<ul>
  <li>شرطی‌سازی بهتر بر طول واژه</li>
  <li>روش‌های هموارسازی (smoothing)</li>
  <li>مدل‌های n-gram با مرتبه‌ی بالاتر</li>
  <li>مدل‌های دوطرفه</li>
</ul>

<b>نکته‌ها:</b>
<ul>
  <li>برای درک بهتر رفتار مدل، می‌توانید چند تقسیم‌بندی متفاوت از داده را امتحان کنید.</li>
  <li>کد شما باید در زمان معقول اجرا شود؛ پیاده‌سازی‌های خیلی کند برای این تکلیف مناسب نیستند.</li>
</ul>

<b>وظیفه:</b> تابع <code>my_best_decoder</code> را کامل کنید.
</div>

In [ ]:
from collections import defaultdict, Counter
import string
import math

###
# Your answer BEGINS HERE
###

# 5-gram, 4-gram, trigram, bigram, unigram counts
char_fivegram_counts = defaultdict(Counter)
char_fourgram_counts = defaultdict(Counter)
char_trigram_counts = defaultdict(Counter)
char_bigram_counts = defaultdict(Counter)
char_unigram_counts = Counter()
word_length_counts = defaultdict(Counter)

for word in train_words:
    padded = "$" + word + "#"
    L = len(word)
    
    for i in range(len(padded)):
        ch = padded[i]
        if ch not in "$#":
            char_unigram_counts[ch] += 1
            word_length_counts[L][ch] += 1
        
        # bigram
        if i < len(padded)-1:
            char_bigram_counts[padded[i]][padded[i+1]] += 1
        
        # trigram
        if i < len(padded)-2:
            char_trigram_counts[(padded[i], padded[i+1])][padded[i+2]] += 1
        
        # 4-gram
        if i < len(padded)-3:
            char_fourgram_counts[(padded[i], padded[i+1], padded[i+2])][padded[i+3]] += 1
        
        # 5-gram
        if i < len(padded)-4:
            char_fivegram_counts[(padded[i], padded[i+1], padded[i+2], padded[i+3])][padded[i+4]] += 1

V = 26
total_unigrams = sum(char_unigram_counts.values())

# all with laplac smoothing
def fivegram_prob(c1, c2, c3, c4, c5):
    total = sum(char_fivegram_counts[(c1, c2, c3, c4)].values()) + V
    return (char_fivegram_counts[(c1, c2, c3, c4)].get(c5, 0) + 1) / total

def fourgram_prob(c1, c2, c3, c4):
    total = sum(char_fourgram_counts[(c1, c2, c3)].values()) + V
    return (char_fourgram_counts[(c1, c2, c3)].get(c4, 0) + 1) / total

def trigram_prob(c1, c2, c3):
    total = sum(char_trigram_counts[(c1, c2)].values()) + V
    return (char_trigram_counts[(c1, c2)].get(c3, 0) + 1) / total

def bigram_prob(c1, c2):
    total = sum(char_bigram_counts[c1].values()) + V
    return (char_bigram_counts[c1].get(c2, 0) + 1) / total

def unigram_prob(c):
    return (char_unigram_counts.get(c, 0) + 1) / (total_unigrams + V)

def my_best_decoder(mask, guessed):
    word_length = len(mask)
    empty_positions = [i for i, ch in enumerate(mask) if ch == '_']
    
    best_char = None
    best_score = -float('inf')
    
    for char in string.ascii_lowercase:
        if char in guessed:
            continue
        
        score = 0.0
        
        for pos in empty_positions:
            # left context
            left4 = None
            left3 = None
            left2 = None
            left1 = None
            
            if pos == 0:
                left4, left3, left2, left1 = "$", "$", "$", "$"
            elif pos == 1:
                left4, left3, left2 = "$", "$", "$"
                left1 = mask[pos-1] if mask[pos-1] != '_' else None
            elif pos == 2:
                left4, left3 = "$", "$"
                left2 = mask[pos-2] if mask[pos-2] != '_' else None
                left1 = mask[pos-1] if mask[pos-1] != '_' else None
            elif pos == 3:
                left4 = "$"
                left3 = mask[pos-3] if mask[pos-3] != '_' else None
                left2 = mask[pos-2] if mask[pos-2] != '_' else None
                left1 = mask[pos-1] if mask[pos-1] != '_' else None
            else:
                left1 = mask[pos-1] if mask[pos-1] != '_' else None
                left2 = mask[pos-2] if mask[pos-2] != '_' else None
                left3 = mask[pos-3] if mask[pos-3] != '_' else None
                left4 = mask[pos-4] if mask[pos-4] != '_' else None
            
            # right context (for backward model)
            right1 = mask[pos+1] if pos < word_length-1 and mask[pos+1] != '_' else None
            right2 = mask[pos+2] if pos < word_length-2 and mask[pos+2] != '_' else None
            right3 = mask[pos+3] if pos < word_length-3 and mask[pos+3] != '_' else None
            
            # hierarchical backoff model (5-gram -> 4-gram -> trigram -> bigram -> unigram)
            if left1 is not None and left2 is not None and left3 is not None and left4 is not None:
                # use 5-gram when we have 4 previous chars
                p = 0.55 * fivegram_prob(left4, left3, left2, left1, char) + \
                    0.2 * fourgram_prob(left3, left2, left1, char) + \
                    0.1 * trigram_prob(left2, left1, char) + \
                    0.1 * bigram_prob(left1, char) + \
                    0.05 * unigram_prob(char)
            elif left1 is not None and left2 is not None and left3 is not None:
                # use 4-gram when we have 3 previous chars
                p = 0.7 * fourgram_prob(left3, left2, left1, char) + \
                    0.15 * trigram_prob(left2, left1, char) + \
                    0.1 * bigram_prob(left1, char) + \
                    0.05 * unigram_prob(char)
            elif left1 is not None and left2 is not None:
                # use trigram when we have 2 previous chars
                p = 0.65 * trigram_prob(left2, left1, char) + \
                    0.25 * bigram_prob(left1, char) + \
                    0.1 * unigram_prob(char)
            elif left1 is not None:
                # use bigram when we have 1 previous char
                p = 0.7 * bigram_prob(left1, char) + \
                    0.3 * unigram_prob(char)
            else:
                # use unigram when no context
                p = unigram_prob(char)
            
            # backward probability (using future context)
            backward_weight = 1.0
            if right1 is not None:
                backward_weight *= bigram_prob(char, right1)
                if right2 is not None:
                    backward_weight *= (trigram_prob(char, right1, right2) ** 0.3)
                    if right3 is not None:
                        backward_weight *= (fourgram_prob(char, right1, right2, right3) ** 0.15)
            
            p *= backward_weight
            
            score += math.log(max(p, 1e-10))
        
        # normalize by number of blanks
        if empty_positions:
            score /= len(empty_positions)
        
        # length prior
        length_prior = (word_length_counts[word_length].get(char, 0) + 1) / \
                       (sum(word_length_counts[word_length].values()) + V)
        
        score += 0.5 * math.log(length_prior)
        
        if score > best_score:
            best_score = score
            best_char = char
    
    return best_char if best_char else 'e' # most probable character in english

###
# Your answer ENDS HERE
###

result = evaluate_decoder(my_best_decoder, eval_words)
print("Evaluating my best decoder on the full evaluation set")
print("Average number of incorrect guesses:", result)

Evaluating my best decoder on the full evaluation set
Average number of incorrect guesses: 7.152


### <div dir="rtl" style="text-align: right; font-family: Vazir, Tahoma, sans-serif;">سؤال ۷ (۰.۵ نمره)</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
رویکرد خود را در یک پاراگراف کوتاه توضیح دهید و نتیجه‌ی به‌دست‌آمده را به‌اختصار تحلیل کنید.
</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-family: Vazir, Tahoma, sans-serif;">
 برای بخش 6، یک مدل زبانی ترکیبی بر پایه n-gramها پیاده‌سازی کردم که از یونی‌گرام تا 5-گرام با هموارسازی لاپلاس استفاده می‌کند. برای هر موقعیت خالی در کلمه، با در نظر گرفتن حروف چپ و راست و با استفاده از یک مدل backoff وزنی ، امتیاز هر حرف را محاسبه کرده و سپس با اعمال اولویت طول کلمه (با وزن ۰.۵)، بهترین حرف را انتخاب می‌کند. نتایج ارزیابی نشان می‌دهد که این مدل نسبت به مدل های قبل عملکرد بهتری دارد.
</div>